# Convolutional Neural Network - Accident Detection

### Project Description

Road traffic accidents are among the leading causes of injury and death worldwide. Traditional surveillance relies on human operators watching CCTV feeds — a process that is slow, error-prone, and not scalable across city-wide networks. An automated accident detection system powered by Deep Learning can flag incidents in real-time, enabling faster emergency response and potentially saving lives.

In this project, we train and compare three **Convolutional Neural Network (CNN)** pipelines to classify CCTV video frames into two categories:

| Class | Description |
|---|---|
| **Accident** | Frame contains an active road accident (collision, crash, overturn) |
| **Non Accident** | Frame shows normal road or traffic conditions |

---

### Dataset Description

**Source:** [Kaggle — Accident Detection from CCTV Footage](https://www.kaggle.com/datasets/ckay16/accident-detection-from-cctv-footage)  
**Format:** JPEG images extracted from real-world CCTV cameras  
**Total Images:** 990 frames across train, validation, and test splits

| Split | Accident | Non Accident | Total |
|---|---|---|---|
| **Train** | 369 | 422 | **791** |
| **Validation** | 46 | 52 | **98** |
| **Test** | 47 | 54 | **101** |
| **Total** | **462** | **528** | **990** |

**Key characteristics:**
- Images captured from fixed overhead and roadside cameras in real-world conditions
- Variable lighting, camera angles, weather, and image resolution
- All frames resized to **128 x 128 pixels** for training
- Dataset is small (~790 training images) — making transfer learning especially valuable
- Slight class imbalance (Non Accident: 53%, Accident: 47%) — manageable without oversampling

---

### Project Objective

The goal is to build an **automated CCTV accident detection system** that classifies frames as *Accident* or *Non Accident* with high accuracy. Specifically, we aim to:

1. **Build a Deep CNN from Scratch** — 4 convolutional blocks (32 to 256 filters) + a 3-layer dense head (512 to 128 neurons) trained purely on this dataset.
2. **Apply Transfer Learning (MobileNetV2)** — use a frozen ImageNet-pretrained backbone to extract powerful visual features without needing large amounts of data.
3. **Combine Transfer Learning with Data Augmentation** — add random flips, rotations, shifts, and zoom to expand the effective training set and improve generalisation.
4. **Compare all three approaches** on the held-out test set and identify the best model.
5. **Run sample inference** — demonstrate the best model predicting on individual images.

---

### Why CNNs for Image Classification?

- **Local receptive fields:** Filters detect spatial patterns (edges, textures, shapes) that are meaningful for accident detection regardless of their position in the frame.
- **Parameter sharing:** The same filter is applied across the entire image, drastically reducing trainable parameters compared to a fully-connected layer.
- **Hierarchical learning:** Shallow layers detect low-level features (edges, colours); deeper layers combine these into high-level concepts (vehicles, crash geometry).
- **Translation invariance:** A crash on the left or right of the frame triggers the same detector — the model generalises across camera positions.

---

### 1. Importing the Dataset from Kaggle

## 1. Importing Required Libraries
In this cell, we import libraries for image processing, visualization, and building our CNN models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


## 2. Exploratory Data Analysis (EDA) & Visualizations
We will load some sample images from the training dataset to understand what the 'Accident' and 'Non Accident' footage looks like.

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = os.getcwd()

import kaggle
kaggle.api.dataset_download_files(
    'ckay16/accident-detection-from-cctv-footage',
    path='data', unzip=True
)
print('Dataset downloaded and extracted!')

In [ ]:
train_dir = 'data/train'
test_dir = 'data/test'
val_dir = 'data/val'

# Function to plot sample images
def plot_samples(directory, category, num_samples=3):
    path = os.path.join(directory, category)
    if not os.path.exists(path):
        print(f"Directory {path} not found.")
        return
    
    images = os.listdir(path)[:num_samples]
    plt.figure(figsize=(15, 5))
    for i, img_name in enumerate(images, 1):
        img_path = os.path.join(path, img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(1, num_samples, i)
        plt.imshow(img)
        plt.title(f"{category} (Shape: {img.shape})")
        plt.axis('off')
    plt.show()

print("Visualizing 'Accident' Samples:")
plot_samples(train_dir, 'Accident', num_samples=3)

print("Visualizing 'Non Accident' Samples:")
plot_samples(train_dir, 'Non Accident', num_samples=3)


## 3. Data Preprocessing
We will create `ImageDataGenerator` instances to load the images.
For the first two approaches, we will **NOT** use data augmentation, only rescaling (normalization).

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# No augmentation, just rescaling
train_datagen_basic = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator_basic = train_datagen_basic.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)


## 4. Approach 1: CNN Model from Scratch
We will build a simple Convolutional Neural Network from scratch using Conv2D, MaxPooling2D, and Dense layers.

In [ ]:
model_scratch = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_scratch.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_scratch.summary()

# Training
history_scratch = model_scratch.fit(
    train_generator_basic,
    validation_data=val_generator,
    epochs=10, # Reduced for faster execution
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

# Evaluating on Test Set
test_loss_scratch, test_acc_scratch = model_scratch.evaluate(test_generator)
print(f"Scratch Model Test Accuracy: {test_acc_scratch:.4f}")


## 5. Approach 2: Pretrained Model (Transfer Learning)
We will use a pretrained model like `MobileNetV2` (trained on ImageNet) to see if the accuracy improves compared to our scratch model. We will freeze the base layers and add our own classification head.

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model
base_model.trainable = False

model_pretrained = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_pretrained.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Training
history_pretrained = model_pretrained.fit(
    train_generator_basic,
    validation_data=val_generator,
    epochs=10,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

# Evaluating on Test Set
test_loss_pre, test_acc_pre = model_pretrained.evaluate(test_generator)
print(f"Pretrained Model Test Accuracy: {test_acc_pre:.4f}")


## 6. Approach 3: Pretrained Model WITH Data Augmentation
Now, we will introduce Data Augmentation into our training pipeline and retrain our pretrained model to see if it further improves robustness and accuracy.

In [ ]:
# Creating Augmented Data Generator
train_datagen_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator_aug = train_datagen_aug.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# Re-initialize the pretrained model to train from scratch with augmented data
base_model_aug = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model_aug.trainable = False

model_aug = Sequential([
    base_model_aug,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_aug.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train with ModelCheckpoint to save the absolute best model for Streamlit
checkpoint_aug = ModelCheckpoint('best_cnn_model.keras', save_best_only=True, monitor='val_accuracy', mode='max')

history_aug = model_aug.fit(
    train_generator_aug,
    validation_data=val_generator,
    epochs=15,
    callbacks=[EarlyStopping(patience=4, restore_best_weights=True), checkpoint_aug]
)

# Evaluating on Test Set
test_loss_aug, test_acc_aug = model_aug.evaluate(test_generator)
print(f"Augmented Model Test Accuracy: {test_acc_aug:.4f}")


## 7. Comparison and Conclusion
Comparing the training and testing accuracy of all 3 approaches.

In [ ]:
print("==== Model Comparison ====")
print(f"1. Model from Scratch        - Test Accuracy: {test_acc_scratch:.4f}")
print(f"2. Pretrained Model          - Test Accuracy: {test_acc_pre:.4f}")
print(f"3. Pretrained + Augmentation - Test Accuracy: {test_acc_aug:.4f}")

# Plotting the comparison
labels = ['Scratch', 'Pretrained', 'Pretrained + Aug']
accs = [test_acc_scratch, test_acc_pre, test_acc_aug]

plt.figure(figsize=(8, 5))
sns.barplot(x=labels, y=accs, palette='viridis')
plt.ylim(0, 1)
plt.title('Test Accuracy Comparison')
plt.ylabel('Accuracy')
plt.show()

# Suggesting the best model
best_acc = max(accs)
if best_acc == test_acc_aug:
    print("\nConclusion: The Pretrained model with Data Augmentation performed the best.")
elif best_acc == test_acc_pre:
    print("\nConclusion: The Pretrained model without Data Augmentation performed the best.")
else:
    print("\nConclusion: The Scratch model surprisingly performed the best.")
